In [1]:
from recbole.quick_start import run_recbole

from itertools import product
from pathlib import Path

import pandas as pd
import json

/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ray/_private/parameter.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [16]:
import os
from pathlib import Path

# Required only for loading checkpoints generated by a trusted local run.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from recbole.quick_start import run_recbole
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed


def find_project_root(start: Path) -> Path:
    for directory in (start, *start.parents):
        if (directory / "run_recbole.py").exists():
            return directory
    raise FileNotFoundError("Could not find project root (run_recbole.py)")


ROOT = find_project_root(Path.cwd().resolve())
DATASET = "ml-100k"
SEED = 2020
CONFIG_ROOT = ROOT / "recbole" / "config"
DATA_DIR = ROOT / "dataset"
CHECKPOINT_DIR = ROOT / "saved"
RESULTS_DIR = ROOT / "group_18" / "results" / "task_1_2"
RESULTS_FILE = RESULTS_DIR / "baseline_results.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed


def evaluate_traditional_model(model_name, config_path, params):
    """Fit a traditional RecBole model and evaluate on validation data."""

    config = Config(
        model=model_name,
        dataset="ml-100k",
        config_file_list=[str(config_path)],
        config_dict={
            "seed": 2020,
            "reproducibility": True,
            "use_gpu": False,
            "data_path": str(DATA_DIR),
            **params,
        },
    )

    init_seed(config["seed"], config["reproducibility"])

    dataset = create_dataset(config)
    train_data, valid_data, _ = data_preparation(config, dataset)

    # EASE and SLIMElastic fit their parameters during initialization
    model = get_model(model_name)(
        config,
        train_data.dataset
    ).to(config["device"])

    trainer = get_trainer(
        config["MODEL_TYPE"],
        config["model"]
    )(config, model)

    valid_result = trainer.evaluate(
        valid_data,
        load_best_model=False
    )

    return {
        "best_valid_score": valid_result["mrr@10"],
        "best_valid_result": valid_result,
    }

In [4]:
def tune_model(
    name,
    model,
    config_path,
    search_space,
    fixed_params=None,
    save=True
):
    """
    Exhaustively tune a RecBole model.

    Parameters
    ----------
    name : str
        Name used for displaying/saving results.

    model : str
        RecBole model name.

    config_path : str or Path
        Path to the model's YAML configuration.

    search_space : dict
        Hyperparameters and values to test.

        Example:
        {
            "k": [20, 50, 100],
            "shrink": [0, 10, 50]
        }

    fixed_params : dict, optional
        Parameters that should remain fixed.
        Useful for things such as knn_method.

    save : bool
        Save tuning results to CSV.

    Returns
    -------
    DataFrame
        All tuning experiments.
    """

    if fixed_params is None:
        fixed_params = {}

    # Parameter names
    param_names = list(search_space.keys())

    # All combinations
    combinations = list(
        product(*search_space.values())
    )

    results = []

    print(f"Tuning {name}")
    print(f"Number of experiments: {len(combinations)}")
    print("=" * 60)

    for i, values in enumerate(combinations, start=1):

        params = dict(zip(param_names, values))

        config_dict = {
            "model": model,
            "dataset": "ml-100k",
            **fixed_params,
            **params,
        }

        print(
            f"[{i}/{len(combinations)}] "
            f"{name}: {params}"
        )

        if model in {"EASE", "SLIMElastic"}:
            result = evaluate_traditional_model(
                model_name=model,
                config_path=config_path,
                params={**fixed_params, **params},
            )
        else:
            result = run_recbole(
                model=model,
                dataset="ml-100k",
                config_dict=config_dict,
                config_file_list=[str(config_path)],
                saved=False,
            )

        valid = result["best_valid_result"]

        row = {
            **params,
            "valid_score": result["best_valid_score"],
            **valid,
        }

        results.append(row)
        
        valid = result["best_valid_result"]

    df = pd.DataFrame(results)

    # Best first
    df = df.sort_values(
        "valid_score",
        ascending=False
    ).reset_index(drop=True)

    if save:
        output_path = RESULTS_DIR / f"{name}_tuning.csv"
        df.to_csv(output_path, index=False)

        print(f"\nSaved to:")
        print(output_path)

    print("\nBest configuration:")
    display(df.head(1))

    return df

In [5]:
itemcf_space = {
    "k": [20, 50, 100, 200],
    "shrink": [0, 10, 50, 100]
}

In [6]:
itemcf_results = tune_model(
    name="ItemCF",
    model="ItemKNN",
    config_path=CONFIG_ROOT / "ItemKNN" / "ml-100k.yaml",
    search_space=itemcf_space,
    fixed_params={
        "knn_method": "item"
    }
)

Tuning ItemCF
Number of experiments: 16
[1/16] ItemCF: {'k': 20, 'shrink': 0}


24 Oct 13:59    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 13:59    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[2/16] ItemCF: {'k': 20, 'shrink': 10}


24 Oct 13:59    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 13:59    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[3/16] ItemCF: {'k': 20, 'shrink': 50}


...

24 Oct 14:00    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:00    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[16/16] ItemCF: {'k': 200, 'shrink': 100}


24 Oct 14:00    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:00    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =


Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/group_18/results/task_1_2/ItemCF_tuning.csv

Best configuration:


,k,shrink,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,100,100,0.3869,0.2007,0.3869,0.2216,0.7264,0.1496


In [7]:
usercf_results = tune_model(
    name="UserCF",
    model="ItemKNN",
    config_path=CONFIG_ROOT / "ItemKNN" / "ml-100k.yaml",
    search_space=itemcf_space,
    fixed_params={
        "knn_method": "user"
    }
)

Tuning UserCF
Number of experiments: 16
[1/16] UserCF: {'k': 20, 'shrink': 0}


24 Oct 14:00    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:00    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[2/16] UserCF: {'k': 20, 'shrink': 10}


24 Oct 14:00    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:00    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[3/16] UserCF: {'k': 20, 'shrink': 50}


...

24 Oct 14:01    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:01    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =

[16/16] UserCF: {'k': 200, 'shrink': 100}


24 Oct 14:01    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:01    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm =


Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/group_18/results/task_1_2/UserCF_tuning.csv

Best configuration:


,k,shrink,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,50,0,0.401,0.216,0.401,0.2355,0.755,0.1562


In [8]:
bpr_space = {
    "embedding_size": [16, 32, 64, 128],
    "learning_rate": [0.0001, 0.001, 0.01],
}

In [9]:
bpr_results = tune_model(
    name="BPR",
    model="BPR",
    config_path=CONFIG_ROOT / "BPR" / "ml-100k.yaml",
    search_space=bpr_space
)

Tuning BPR
Number of experiments: 12
[1/12] BPR: {'embedding_size': 16, 'learning_rate': 0.0001}


24 Oct 14:01    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:01    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.0001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm 

[2/12] BPR: {'embedding_size': 16, 'learning_rate': 0.001}


24 Oct 14:01    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:01    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[3/12] BPR: {'embedding_size': 16, 'learning_rate': 0.01}


...

24 Oct 14:02    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:02    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[12/12] BPR: {'embedding_size': 128, 'learning_rate': 0.01}


24 Oct 14:02    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:02    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.01
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm = 


Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/group_18/results/task_1_2/BPR_tuning.csv

Best configuration:


,embedding_size,learning_rate,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,32,0.01,0.3581,0.1904,0.3581,0.2038,0.7169,0.1393


In [10]:
slim_space = {
    "alpha": [0.0001, 0.001, 0.01, 0.1],
    "l1_ratio": [0.1, 0.5, 0.9]
}

slim_results = tune_model(
    name="SLIM",
    model="SLIMElastic",
    config_path=CONFIG_ROOT / "SLIMElastic" / "ml-100k.yaml",
    search_space=slim_space
)

Tuning SLIM
Number of experiments: 12
[1/12] SLIM: {'alpha': 0.0001, 'l1_ratio': 0.1}


24 Oct 14:02    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:02    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[2/12] SLIM: {'alpha': 0.0001, 'l1_ratio': 0.5}


24 Oct 14:02    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:02    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[3/12] SLIM: {'alpha': 0.0001, 'l1_ratio': 0.9}


...

24 Oct 14:04    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:04    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[12/12] SLIM: {'alpha': 0.1, 'l1_ratio': 0.9}


24 Oct 14:04    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:04    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]



Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/group_18/results/task_1_2/SLIM_tuning.csv

Best configuration:


,alpha,l1_ratio,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,0.1,0.1,0.4165,0.2271,0.4165,0.2495,0.7625,0.1709


In [11]:
ease_space = {
    "reg_weight": [10, 50, 100, 200, 500, 1000]
}

ease_results = tune_model(
    name="EASE",
    model="EASE",
    config_path=CONFIG_ROOT / "EASE" / "ml-100k.yaml",
    search_space=ease_space
)

Tuning EASE
Number of experiments: 6
[1/6] EASE: {'reg_weight': 10}


24 Oct 14:04    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:04    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[2/6] EASE: {'reg_weight': 50}


24 Oct 14:04    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:04    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[3/6] EASE: {'reg_weight': 100}


...

24 Oct 14:04    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:04    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]


[6/6] EASE: {'reg_weight': 1000}


24 Oct 14:04    INFO  [Training]: train_batch_size = [2048] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
24 Oct 14:04    INFO  [Evaluation]: eval_batch_size = [4096] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'order': 'RO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}]



Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/group_18/results/task_1_2/EASE_tuning.csv

Best configuration:


,reg_weight,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,500,0.4189,0.2386,0.4189,0.2568,0.7815,0.1766


In [17]:
fism_space = {
    "embedding_size": [32, 64],
    "learning_rate": [0.001, 0.005],
    "alpha": [0.5],
    "reg_weights": [
        [0.0001, 0.0001],
        [0.001, 0.001],
    ],
}

fism_results = tune_model(
    name="FISM",
    model="FISM",
    config_path=CONFIG_ROOT / "FISM" / "ml-100k.yaml",
    search_space=fism_space,
)

Tuning FISM
Number of experiments: 8
[1/8] FISM: {'embedding_size': 32, 'learning_rate': 0.001, 'alpha': 0.5, 'reg_weights': [0.0001, 0.0001]}


24 Oct 14:10    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:10    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[2/8] FISM: {'embedding_size': 32, 'learning_rate': 0.001, 'alpha': 0.5, 'reg_weights': [0.001, 0.001]}


24 Oct 14:13    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:13    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[3/8] FISM: {'embedding_size': 32, 'learning_rate': 0.005, 'alpha': 0.5, 'reg_weights': [0.0001, 0.0001]}


...

24 Oct 14:29    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:29    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =

[8/8] FISM: {'embedding_size': 64, 'learning_rate': 0.005, 'alpha': 0.5, 'reg_weights': [0.001, 0.001]}


24 Oct 14:33    INFO  ['/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3c9b1d4a1e8ef3e34af002dd35dea90e2dc9ab7db.json']
24 Oct 14:33    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/.venv/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 20
train_batch_size = 2048
learner = adam
learning_rate = 0.005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 10
stopping_step = 20
clip_grad_norm =


Saved to:
/Users/randy/Desktop/Recommender Systems/New Final Project/RecBole_DSAIT4335_Group_18/group_18/results/task_1_2/FISM_tuning.csv

Best configuration:


,embedding_size,learning_rate,alpha,reg_weights,valid_score,recall@10,mrr@10,ndcg@10,hit@10,precision@10
0,64,0.005,0.5,"[0.0001, 0.0001]",0.3429,0.1821,0.3429,0.1976,0.7169,0.1417
